# 🌊 Sea Level Monitoring Pipeline
## Copernicus CDS → ZIP → Parquet → DuckDB

**Note:** CDS renvoie un archive ZIP avec un fichier .nc par mois

## Setup avec `uv`

```bash
mkdir sea-level-monitoring && cd sea-level-monitoring
uv init --python 3.11
uv add earthkit cdsapi xarray pandas pyarrow matplotlib numpy duckdb netcdf4
uv add --dev jupyter ipython
source .venv/bin/activate
jupyter notebook
```

## 1️⃣ Import & Setup

In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import duckdb
import warnings
import zipfile
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')
plt.style.use('catppuccin-macchiato')

data_dir = Path('./data')
data_dir.mkdir(exist_ok=True)

print('✅ Ready!')

## 2️⃣ Download Sea Level Data (ZIP Archive)

**Important:** CDS retourne un ZIP avec un .nc par mois

In [ ]:
import cdsapi

zip_file = data_dir / 'sea_level_data.zip'

# Petit dataset de test (2023-2024)
request = {
    'product_type': 'monthly_mean_sea_level_anomaly',
    'variable': 'sea_level_anomaly',
    'year': ['2023', '2024'],
    'month': ['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12'],
    'format': 'netcdf'
}

print('📥 Téléchargement depuis Copernicus CDS...')
print(f'Période: 2023-2024 (24 fichiers .nc dans un ZIP)')
print(f'Output: {zip_file}')
print('\n⏳ Cela prend 2-5 minutes...\n')

try:
    client = cdsapi.Client(timeout=600)
    client.retrieve(
        'satellite-sea-level-global',
        request,
        str(zip_file)
    )
    print('✅ Téléchargement complété!')
    size_mb = zip_file.stat().st_size / 1024 / 1024
    print(f'📁 Taille ZIP: {size_mb:.2f} MB')
    
except Exception as e:
    print(f'❌ Erreur: {e}')
    print('\nFix:')
    print('1. Vérifiez ~/.cdsapirc existe')
    print('2. Lancez: chmod 600 ~/.cdsapirc')
    print('3. Supprimez le fichier: rm data/sea_level_data.zip')
    print('4. Réessayez')

## 3️⃣ Extract ZIP Archive

In [ ]:
nc_dir = data_dir / 'nc_files'
nc_dir.mkdir(exist_ok=True)

print('📦 Extraction du ZIP...\n')

try:
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(nc_dir)
    
    # Lister les fichiers .nc
    nc_files = sorted(list(nc_dir.glob('*.nc')))
    print(f'✅ Extraction complétée!')
    print(f'\n📊 Fichiers extraits: {len(nc_files)}')
    for f in nc_files[:5]:  # Afficher les 5 premiers
        print(f'  - {f.name}')
    if len(nc_files) > 5:
        print(f'  ... et {len(nc_files) - 5} autres')
        
except Exception as e:
    print(f'❌ Erreur: {e}')

## 4️⃣ Load & Combine All NC Files

In [ ]:
nc_dir = data_dir / 'nc_files'
nc_files = sorted(list(nc_dir.glob('*.nc')))

print(f'🔓 Chargement de {len(nc_files)} fichiers NetCDF...\n')

datasets = []
for nc_file in nc_files:
    try:
        ds = xr.open_dataset(nc_file, engine='netcdf4')
        datasets.append(ds)
        print(f'✅ {nc_file.name}')
    except Exception as e:
        print(f'❌ {nc_file.name}: {e}')

if datasets:
    # Combiner tous les datasets selon le temps
    print('\n🔀 Combinaison des datasets...')
    ds = xr.concat(datasets, dim='time').sortby('time')
    print(f'✅ Combined dataset shape: {dict(ds.dims)}')
    print(f'\n{ds}')
else:
    print('❌ Aucun fichier chargé')

## 5️⃣ Explore Data Quality

In [ ]:
print('='*50)
print('📊 Statistiques:\n')
for var in ds.data_vars:
    print(f'{var}:')
    data = ds[var].values.flatten()
    data = data[~np.isnan(data)]  # Remove NaNs
    print(f'  Mean: {np.mean(data):.6f}')
    print(f'  Std: {np.std(data):.6f}')
    print(f'  Min: {np.min(data):.6f}')
    print(f'  Max: {np.max(data):.6f}')
    print(f'  Median: {np.median(data):.6f}\n')


## 6️⃣ Visualisations

In [ ]:
if 'sla' in ds.data_vars:
    print('📈 Création de la carte...')
    
    try:
        latest = ds['sla'].isel(time=-1)
        
        fig, ax = plt.subplots(figsize=(14, 8))
        latest.plot(ax=ax, cmap='RdBu_r', robust=True,
                    cbar_kwargs={'label': 'Anomalie (m)'})
        
        time_str = pd.Timestamp(latest.time.values).strftime('%Y-%m')
        ax.set_title(f'Anomalie de niveau marin - {time_str}', fontsize=14, fontweight='bold')
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        
        plt.tight_layout()
        plt.savefig(data_dir / 'map.png', dpi=300)
        plt.show()
        
        print('✅ Sauvegardé: map.png')
    except Exception as e:
        print(f'❌ Erreur: {e}')
        print('Vérifiez que sla contient des données valides')


In [ ]:
# Time series globale
if 'sla' in ds.data_vars:
    print('📈 Création de la série temporelle...')
    
    try:
        global_mean = ds['sla'].mean(dim=['longitude', 'latitude'])
        
        fig, ax = plt.subplots(figsize=(12, 6))
        global_mean.plot(ax=ax, linewidth=2.5, marker='o', markersize=5, color='steelblue')
        
        # Ligne de tendance - éviter les opérations sur datetime
        x = np.arange(len(global_mean))
        y = global_mean.values
        z = np.polyfit(x, y, 1)
        p = np.poly1d(z)
        ax.plot(x, p(x), 'r--', linewidth=2, label='Tendance')
        
        ax.set_title('Anomalie moyenne globale de niveau marin', fontsize=14, fontweight='bold')
        ax.set_xlabel('Date')
        ax.set_ylabel('Anomalie (m)')
        ax.grid(True, alpha=0.3)
        ax.legend()
        
        plt.tight_layout()
        plt.savefig(data_dir / 'timeseries.png', dpi=300)
        plt.show()
        
        print('✅ Sauvegardé: timeseries.png')
    except Exception as e:
        print(f'❌ Erreur: {e}')
        import traceback
        traceback.print_exc()


## 7️⃣ Transform to Parquet

In [ ]:
print('🔄 Conversion en Parquet...\n')

if 'sla' in ds.data_vars:
    # Convertir en DataFrame
    sla = ds['sla'].to_pandas().reset_index()
    df = sla.melt(id_vars=['time'], var_name='location', value_name='sea_level_anomaly_m')
    
    # Extraire coordonnées
    df['latitude'] = df['location'].apply(lambda x: float(str(x).split(',')[0].replace('(', '')))
    df['longitude'] = df['location'].apply(lambda x: float(str(x).split(',')[1].replace(')', '')))
    
    # Colonnes datetime - IMPORTANT: convert timestamp to string for Parquet
    df = df.drop('location', axis=1)
    df['timestamp'] = df['time'].astype(str)  # Convert datetime64 to string
    df['year'] = pd.to_datetime(df['time']).dt.year
    df['month'] = pd.to_datetime(df['time']).dt.month
    df = df.dropna(subset=['sea_level_anomaly_m'])
    
    df = df[['timestamp', 'year', 'month', 'latitude', 'longitude', 'sea_level_anomaly_m']]
    
    print(f'📊 Shape: {df.shape}')
    print(f'\n🔍 Exemple:')
    print(df.head(10))
    print(f'\nDtypes:')
    print(df.dtypes)
    
    # Sauvegarder
    parquet_file = data_dir / 'sea_level_data.parquet'
    df.to_parquet(parquet_file, compression='snappy', engine='pyarrow', index=False)
    
    print(f'\n✅ Sauvegardé: {parquet_file}')
    print(f'Taille: {parquet_file.stat().st_size / 1024 / 1024:.2f} MB')


## 8️⃣ Load into DuckDB

In [ ]:
parquet_file = data_dir / 'sea_level_data.parquet'
db_file = data_dir / 'sea_level.duckdb'

print('🗄️  Chargement dans DuckDB...\n')

conn = duckdb.connect(str(db_file))

try:
    conn.execute('DROP TABLE IF EXISTS sea_level')
except:
    pass

conn.execute(
    f"CREATE TABLE sea_level AS SELECT * FROM read_parquet('{parquet_file}')"
)

result = conn.execute('SELECT COUNT(*) FROM sea_level').fetchall()
print(f'✅ Table créée')
print(f'Lignes: {result[0][0]:,}')

schema = conn.execute('DESCRIBE sea_level').fetchall()
print('\n📋 Colonnes:')
for col, dtype, *_ in schema:
    print(f'  {col}: {dtype}')

## 9️⃣ SQL Queries

In [ ]:
# Query 1: Moyenne par année
print('📊 Anomalie moyenne par année:\n')

df_result = conn.execute('''
    SELECT
        year,
        COUNT(*) as measurements,
        ROUND(AVG(sea_level_anomaly_m), 4) as mean_m,
        ROUND(MIN(sea_level_anomaly_m), 4) as min_m,
        ROUND(MAX(sea_level_anomaly_m), 4) as max_m
    FROM sea_level
    GROUP BY year
    ORDER BY year
''').df()

print(df_result.to_string(index=False))

In [ ]:
# Query 2: Emplacements extrêmes
print('\n📊 Anomalies les plus hautes (dernier mois):\n')

# D'abord vérifier les données
latest = conn.execute('SELECT MAX(timestamp) FROM sea_level').fetchall()
print(f'Latest timestamp: {latest[0][0]}\n')

df_result = conn.execute('''
    SELECT
        ROUND(latitude, 2) as lat,
        ROUND(longitude, 2) as lon,
        ROUND(sea_level_anomaly_m, 4) as anomaly_m,
        timestamp
    FROM sea_level
    WHERE timestamp = (SELECT MAX(timestamp) FROM sea_level)
    ORDER BY sea_level_anomaly_m DESC
    LIMIT 10
''').df()

if len(df_result) > 0:
    print(df_result.to_string(index=False))
else:
    print('Aucune donnée trouvée pour le dernier mois')


## ✨ Terminé!

**Pipeline:** CDS ZIP → Extraction → Parquet → DuckDB → SQL

**Prochaines étapes:** Passer à l'ensemble complet 2019-2024 & ajouter les modèles ML